# Woche 12 – Montag: Wiederholung Risikomaße & Autoencoder (Woche 9)

## Block 1 (06:00 – 08:45): VaR, ES, Kohärenz

**Aufgabe 1:** Definieren Sie den Value‑at‑Risk (VaR) zum Konfidenzniveau $\alpha$ und den Expected Shortfall (ES).

**Aufgabe 2:** Erklären Sie, warum ES kohärent ist und VaR nicht (fehlende Subadditivität).

**Aufgabe 3:** Berechnen Sie VaR (95%) und ES für einen Datensatz Ihrer Wahl (z. B. normalverteilte Verluste) mit Python.

> **Verbesserungshinweis:** Der Expected Shortfall ist der Durchschnitt der Verluste, die den VaR überschreiten: $\text{ES}_\alpha = \mathbb{E}[L \mid L \ge \text{VaR}_\alpha]$.


In [ ]:
import numpy as np

np.random.seed(42)
losses = np.random.normal(0, 1, 1000)
alpha = 0.95
var = np.percentile(losses, alpha * 100)
es = losses[losses >= var].mean()
print(f"VaR (95%): {var:.4f}, ES: {es:.4f}")


---

## Block 2 (09:30 – 11:40): Copulas – Wiederholung

**Aufgabe 1:** Was ist eine Copula? Formulieren Sie das Theorem von Sklar.

**Aufgabe 2:** Simulieren Sie eine Gauß‑Copula mit $\rho = 0.5$ und plotten Sie die Scatter‑Matrix.


In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt

rho = 0.5
Sigma = np.array([[1, rho], [rho, 1]])
L = np.linalg.cholesky(Sigma)
z = np.random.normal(0, 1, (1000, 2))
x = z @ L.T
u = stats.norm.cdf(x)
plt.scatter(u[:,0], u[:,1], alpha=0.5)
plt.title(f'Gauß-Copula mit ρ = {rho}')
plt.show()


---

## Block 3 (12:40 – 14:10): Autoencoder – Rekonstruktionsfehler & Anomalieerkennung

**Aufgabe:** Laden Sie Ihren gespeicherten Autoencoder aus Woche 9 (oder trainieren Sie einen neuen) und berechnen Sie den Rekonstruktionsfehler für Ihre AML‑Daten. Identifizieren Sie die Top‑5 Anomalien.


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Daten vorbereiten (Beispiel)
df = pd.read_json('cleaned_aml_batch.json')
features = df[['amount']].copy()
features['amount_log'] = np.log1p(features['amount'])
scaler = StandardScaler()
X = scaler.fit_transform(features)
X_tensor = torch.tensor(X, dtype=torch.float32)

# Autoencoder Definition (muss mit dem trainierten Modell übereinstimmen)
class AE(nn.Module):
    def __init__(self, input_dim=2, bottleneck_dim=1):
        super().__init__()
        self.encoder = nn.Linear(input_dim, bottleneck_dim)
        self.decoder = nn.Linear(bottleneck_dim, input_dim)
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = AE()
try:
    model.load_state_dict(torch.load('best_ae_model.pth'))
    print('Modell geladen')
except:
    print('Kein gespeichertes Modell gefunden – bitte trainieren Sie zuerst ein Modell (siehe Woche 9).')

model.eval()
with torch.no_grad():
    rec = model(X_tensor)
    errors = torch.mean((X_tensor - rec)**2, dim=1).numpy()
df['reconstruction_error'] = errors
top5 = df.nlargest(5, 'reconstruction_error')[['amount', 'reconstruction_error']]
print(top5)


> **Verbesserungshinweis:** Ein hoher Rekonstruktionsfehler deutet auf eine Anomalie hin. In der Praxis sollten diese Fälle manuell überprüft werden.

---

## Block 4 (14:40 – 16:50): SPR – Wiederholung Zustandspassiv & Passiv‑Ersatzformen

**Übung:** Bilden Sie 10 Sätze über Risikomaße und Autoencoder im Zustandspassiv („Der VaR ist berechnet worden“) und mit Passiv‑Ersatzformen („Der ES lässt sich leicht bestimmen“).

**Schreibübung:** Verfassen Sie einen kurzen Bericht (1/2 Seite) über die Ergebnisse Ihrer Anomalieerkennung, ausschließlich im Zustandspassiv. Speichern Sie als `67_wiederholung_risikomasse_autoencoder.md` in Track_C.